In [0]:
    from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from sklearn.impute import KNNImputer
from pyspark.sql.types import DoubleType


In [0]:
mds = spark.read.table("hive_metastore.default.mds")
display(mds)

Databricks visualization. Run in Databricks to view.

In [0]:
mds = mds.select("ID_MDS","ID_OBJECTO","CONCELHO","AO","NOME","TIPOINST","TAGCOM","REDE","CHAVE_SIT","X_SIT","Y_SIT","MARCA","MODELO")

In [0]:
dbutils.data.summarize(mds)

In [0]:
# Path to weather_data location
base_path = "dbfs:/user/hive/warehouse/weather_data.db"

# List of table names (you can also automate this if needed)
locations = [
    "alagoa", "alcacovas", "barragem_de_castelo_burgoes", "rebordelo", 
    "barragem_do_divor", "barragem_do_roxo", "batalha", "campo_experimental_crato", 
    "caxarias", "colares", "comporta", "vila_nova_de_cerveira", "gondizalves", "junqueira", 
    "minas_de_jales", "proenca_a_nova", "santarem", "sao_bras_de_alportel"
]

# Optional: set this to True if you still want to keep track of the original location
add_location_column = True

# Read and union all DataFrames
dfs = []
for loc in locations:
    df = spark.read.format("delta").load(os.path.join(base_path, loc))
    if add_location_column:
        df = df.withColumn("location", lit(loc))
    dfs.append(df)

weather_df = reduce(DataFrame.unionByName, dfs)

# Show schema or a sample
display(weather_df)

In [0]:
weather_df = weather_df.withColumn("date", to_timestamp("date", "dd/MM/yyyy HH:mm"))

display(weather_df)

In [0]:
weather_columns = ["temperatura_media_do_ar_diaria_c", "humidade_relativa_media_diaria_percent", "precipitacao_diaria_mm", "velocidade_do_vento_media_diaria_m/s"]

# Cast each to double
for col_name in weather_columns:
    weather_df = weather_df.withColumn(col_name, col(col_name).cast(DoubleType()))

In [0]:
dbutils.data.summarize(weather_df)

In [0]:
weather_df.filter(col("Date").isNull()).display()

In [0]:
weather_df = weather_df.filter(col("Date").isNotNull())

In [0]:
display(weather_df.filter(col("location") == "rebordelo"))

In [0]:
features = ["temperatura_media_do_ar_diaria_c", "humidade_relativa_media_diaria_percent", "precipitacao_diaria_mm", "velocidade_do_vento_media_diaria_m/s"]

# For each column, calculate % of nulls per location
exprs = [
    (count(when(col(c).isNull(), c)) / count("*")).alias(f"{c}_null_pct")
    for c in features
]

null_stats = weather_df.groupBy("location").agg(*exprs)

display(null_stats)

In [0]:
w_station = Window.partitionBy("location").orderBy("DATE").rowsBetween(-3, 3)

In [0]:
weather_df_clean = weather_df.withColumn(
    "temperatura_media_do_ar_diaria",
    when(col("temperatura_media_do_ar_diaria_c").isNull(), avg("temperatura_media_do_ar_diaria_c").over(w_station))
    .otherwise(col("temperatura_media_do_ar_diaria_c"))
)

Use nearest location to help input missing values

In [0]:
# Define rolling window
rolling_window = Window.partitionBy("location").orderBy("date").rowsBetween(-3, 3)


# === 1. Rolling fill ===
weather_df_clean = weather_df.withColumn(
    "temperatura_media_do_ar_diaria",
    when(col("temperatura_media_do_ar_diaria_c").isNull(), avg("temperatura_media_do_ar_diaria_c").over(rolling_window))
    .otherwise(col("temperatura_media_do_ar_diaria_c"))
)

weather_df_clean = weather_df_clean.withColumn(
    "humidade_relativa_media_diaria",
    when(col("humidade_relativa_media_diaria_percent").isNull(), avg("humidade_relativa_media_diaria_percent").over(rolling_window))
    .otherwise(col("humidade_relativa_media_diaria_percent"))
)

weather_df_clean = weather_df_clean.withColumn(
    "velocidade_do_vento_media_diaria",
    when(col("velocidade_do_vento_media_diaria_m/s").isNull(), avg("velocidade_do_vento_media_diaria_m/s").over(rolling_window))
    .otherwise(col("velocidade_do_vento_media_diaria_m/s"))
)

# Precipitation is not filled via rolling

# === 2. Spatial fill (self-join on date) ===
df_a = weather_df_clean.alias("a")
df_b = weather_df_clean.alias("b")

joined = df_a.join(
    df_b,
    (col("a.date") == col("b.date")) &
    (col("a.location") != col("b.location")),
    how="inner"
)

# Compute distance
joined = joined.withColumn("distance", sqrt(
    pow(col("a.latitude") - col("b.latitude"), 2) +
    pow(col("a.longitude") - col("b.longitude"), 2)
))

w = Window.partitionBy("a.location", "a.date").orderBy("distance")

# Nearest temperature
temperature_spatial = joined.filter(
    col("a.temperatura_media_do_ar_diaria").isNull() & col("b.temperatura_media_do_ar_diaria").isNotNull()
).withColumn("row", row_number().over(w)).filter("row = 1") \
.select(col("a.location"), col("a.date"), col("b.temperatura_media_do_ar_diaria").alias("temperatura_media_do_ar_diaria_spatial"))

# Nearest humidity
humidity_spatial = joined.filter(
    col("a.humidade_relativa_media_diaria").isNull() & col("b.humidade_relativa_media_diaria").isNotNull()
).withColumn("row", row_number().over(w)).filter("row = 1") \
.select(col("a.location"), col("a.date"), col("b.humidade_relativa_media_diaria").alias("humidade_relativa_media_diaria_spatial"))

# Nearest wind
wind_spatial = joined.filter(
    col("a.velocidade_do_vento_media_diaria").isNull() & col("b.velocidade_do_vento_media_diaria").isNotNull()
).withColumn("row", row_number().over(w)).filter("row = 1") \
.select(col("a.location"), col("a.date"), col("b.velocidade_do_vento_media_diaria").alias("wind_spatial"))

# Nearest precipitation
precip_spatial = joined.filter(
    col("a.precipitacao_diaria_mm").isNull() & col("b.precipitacao_diaria_mm").isNotNull()
).withColumn("row", row_number().over(w)).filter("row = 1") \
.select(col("a.location"), col("a.date"), col("b.precipitacao_diaria_mm").alias("precip_spatial"))

# === 3. Join and finalize ===
df = weather_df_clean.join(temperature_spatial, on=["location", "date"], how="left") \
    .join(humidity_spatial, on=["location", "date"], how="left") \
    .join(wind_spatial, on=["location", "date"], how="left") \
    .join(precip_spatial, on=["location", "date"], how="left")

# Final columns
df = df.withColumn(
    "temperatura_media_do_ar_diaria",
    when(col("temperatura_media_do_ar_diaria").isNotNull(), col("temperatura_media_do_ar_diaria"))
    .when(col("temperatura_media_do_ar_diaria_spatial").isNotNull(), col("temperatura_media_do_ar_diaria_spatial"))
    .otherwise(col("temperatura_media_do_ar_diaria"))
)

df = df.withColumn(
    "humidade_relativa_media_diaria",
    when(col("humidade_relativa_media_diaria").isNotNull(), col("humidade_relativa_media_diaria"))
    .when(col("humidade_relativa_media_diaria_spatial").isNotNull(), col("humidade_relativa_media_diaria_spatial"))
    .otherwise(col("humidade_relativa_media_diaria"))
)

df = df.withColumn(
    "velocidade_do_vento_media_diaria",
    when(col("velocidade_do_vento_media_diaria").isNotNull(), col("velocidade_do_vento_media_diaria"))
    .when(col("wind_spatial").isNotNull(), col("wind_spatial"))
    .otherwise(col("velocidade_do_vento_media_diaria"))
)

df = df.withColumn(
    "precipitacao_diaria",
    when(col("precip_spatial").isNotNull(), col("precip_spatial"))
    .otherwise(lit(0.0))
)

display(df)

In [0]:
df_final_weather = df.select(
    "location",
    "date",
    "latitude",
    "longitude",
    col("temperatura_media_do_ar_diaria").alias("temperature"),
    col("humidade_relativa_media_diaria").alias("humidity"),
    col("precipitacao_diaria").alias("precipitation"),
    col("velocidade_do_vento_media_diaria").alias("wind_speed")
)

display(df_final_weather)

In [0]:
features = ["temperature", "humidity", "precipitation", "wind_speed"]

# For each column, calculate % of nulls per location
exprs = [
    (count(when(col(c).isNull(), c)) / count("*")).alias(f"{c}_null_pct")
    for c in features
]

null_stats = df_final_weather.groupBy("location").agg(*exprs)

display(null_stats)

In [0]:
display(df_final_weather.filter(col("location") == "campo_experimental_crato"))

In [0]:
dbutils.data.summarize(df_final_weather)

In [0]:
df_final_weather.printSchema()

In [0]:


permanent_table_name = "weather_gold"

df_final_weather.write.format("parquet").saveAsTable(permanent_table_name)